model.named_parameters() 是 PyTorch 中 torch.nn.Module 类的一个核心方法。它返回一个迭代器（Iterator），用于访问模型中所有可学习的参数，并且同时提供参数的名称和参数张量本身。

## 核心功能

当你调用这个方法时，PyTorch 会递归地遍历模型的所有子模块，并返回一个生成器。每次迭代都会产生一个包含两个元素的元组：

- 名称 (name)：字符串类型，表示参数的层级路径（例如 "layer1.conv.weight"）。
- 参数 (param)：torch.Tensor 类型，即具体的权重或偏置数据

In [ ]:
for name, param in model.named_parameters():
    print(f"参数名: {name}, 形状: {param.shape}")

## 为什么需要它？（核心用途）
model.parameters() 只能给你参数本身，但在很多高级应用中，<b>你需要知道这个参数属于哪一层才能做决定。named_parameters() 正是为了解决这个问题</b>

### 用途 A：冻结特定层（迁移学习）
在微调预训练模型（如 ResNet, BERT）时，我们通常希望冻结骨干网络（Backbone），只训练新加的分类头（Classifier）。通过检查 name，我们可以精准地“锁定”并冻结参数。

In [ ]:
# 假设 model 是一个预训练的 ResNet
for name, param in model.named_parameters():
    # 如果参数名包含 'layer1' 或 'layer2'，则冻结它
    if 'layer1' in name or 'layer2' in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

### 用途 B：设置分层学习率
有时候，<b>我们希望骨干网络学得慢一点（小学习率），而新加的分类头学得快一点（大学习率）</b>。这需要利用 name 来筛选参数并分组。

In [ ]:
# 筛选出 backbone 和 classifier 的参数
backbone_params = [p for n, p in model.named_parameters() if 'backbone' in n]
classifier_params = [p for n, p in model.named_parameters() if 'classifier' in n]

# 定义优化器，为不同组设置不同学习率
optimizer = torch.optim.Adam([
    {'params': backbone_params, 'lr': 0.001},    # 骨干网络：学习率小
    {'params': classifier_params, 'lr': 0.01}    # 分类头：学习率大
])

### 用途 C：调试与检查
在训练前，你可能想检查某些层是否被正确初始化，或者在训练中检查某些层是否还在更新。

In [ ]:
print("=== 可训练参数检查 ===")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"正在训练: {name}, 均值: {param.data.mean().item():.4f}")
    else:
        print(f"已冻结: {name}")

In [5]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.bn1 = nn.BatchNorm2d(16)
        self.fc = nn.Linear(100, 10)

    def forward(self, x):
        return self.fc(x)

model = SimpleNet()

print("--- 使用 named_parameters() ---")
for name, param in model.named_parameters():
    # name 示例: "conv1.weight", "bn1.running_mean" (注意：state_dict会有running_mean，但named_parameters通常只返回可学习参数)
    # 实际上 bn1.weight 和 bn1.bias 是可学习的，running_mean 不是
    print(f"名称: {name:<20} | 形状: {str(param.shape):<25} | 可训练: {param.requires_grad}")

# 输出示例:
# 名称: conv1.weight         | 形状: torch.Size([16, 3, 3, 3]) | 可训练: True
# 名称: conv1.bias           | 形状: torch.Size([16])          | 可训练: True
# 名称: bn1.weight           | 形状: torch.Size([16])          | 可训练: True
# 名称: bn1.bias             | 形状: torch.Size([16])          | 可训练: True
# 名称: fc.weight            | 形状: torch.Size([10, 100])     | 可训练: True
# 名称: fc.bias              | 形状: torch.Size([10])          | 可训练: True

--- 使用 named_parameters() ---
名称: conv1.weight         | 形状: torch.Size([16, 3, 3, 3]) | 可训练: True
名称: conv1.bias           | 形状: torch.Size([16])          | 可训练: True
名称: bn1.weight           | 形状: torch.Size([16])          | 可训练: True
名称: bn1.bias             | 形状: torch.Size([16])          | 可训练: True
名称: fc.weight            | 形状: torch.Size([10, 100])     | 可训练: True
名称: fc.bias              | 形状: torch.Size([10])          | 可训练: True
